（一）研究策略主要流程
1.因子获取，数据清洗
2.模型/预测
3.*组合优化；同样的DV, 最大的预期收益
4.风控和执行
5.回测


（二）模型：
CTA, 统计套利，多因子，*强化学习，*财报或新闻情绪驱动


（三）高频因子：主要以各类标的的量价指标为主，每天有较多笔交易（相同标的，日内多笔多空）；因为成交较多，必须考虑执行成本，以及仓位的风险控制。
优化目标 Max objective : Expected Return - execution cost - Risk Penalty。 


（四）低频因子（***主要讨论方向***）：半日频，日频为主。需考虑多方面的因子，这方面需要郎总的建议，我考虑到：资金面的变化（银行，非银隔夜利率），市场动量的变化，曲线形态，RV关系，机构买入卖出统计，股票动向等。
结合到上次郎总的分享，我觉得可以加入（a）：1.筹码分歧指标，量化当前市场的筹码均衡水平，如果筹码都集中在多头或者空头，则就是一种比较脆弱的市场结构；2.反转行情监控指标；

债市可能存在难以量化的很重要的一些指标：比如，市场博弈的东西，突发的新闻（降息预期/国债买卖/市场预期突然变化）。这部分可能要以一种人工干预的超参的方法加入模型。


以下是一个简单的代码示例：

In [ ]:
"""

以下是豆包根据指令”帮我写一个，多因子量化策略，涉及数据清洗，因子构建，因子机器学习预测，组合优化，执行。回测。用python“ 生成的代码:


依赖包：
pip install akshare numpy pandas scipy scikit-learn cvxpy backtrader matplotlib warnings

但这份代码跑不通， 因为数据接口的问题，没关系主要是做一个较完整的策略的非常简单的一个演示


"""


import akshare as ak
import numpy as np
import pandas as pd
import warnings
import datetime
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import cvxpy as cp
import backtrader as bt
import matplotlib.pyplot as plt

# 忽略警告
warnings.filterwarnings('ignore')

# ===================== 1. 数据获取与清洗 =====================
class DataProcessor:
    def __init__(self):
        self.stock_codes = None
        self.raw_data = None
        self.clean_data = None

    # 获取股票列表（以沪深300为例）
    def get_stock_list(self):
        # 获取沪深300成分股
        hs300_df = ak.index_stock_cons_csindex(symbol="000300")
        self.stock_codes = hs300_df['成分券代码'].tolist()[:50]  # 取前50只股票加快运行
        return self.stock_codes

    # 获取股票历史数据
    def get_stock_data(self, start_date="2020-01-01", end_date="2024-01-01"):
        all_data = []
        for code in self.stock_codes[:10]:  # 进一步缩小范围，避免数据量过大
            try:
                # 获取日频数据
                df = ak.stock_zh_a_hist(symbol=code, period="daily", 
                                       start_date=start_date, end_date=end_date, 
                                       adjust="qfq")
                df['code'] = code
                all_data.append(df)
            except:
                continue
        
        self.raw_data = pd.concat(all_data, ignore_index=True)
        # 数据格式转换
        self.raw_data['日期'] = pd.to_datetime(self.raw_data['日期'])
        self.raw_data = self.raw_data.sort_values(by=['code', '日期']).reset_index(drop=True)
        return self.raw_data

    # 数据清洗
    def clean_data(self):
        df = self.raw_data.copy()
        
        # 1. 删除缺失值
        df = df.dropna(subset=['开盘价', '最高价', '最低价', '收盘价', '成交量'])
        
        # 2. 去除异常值（3倍标准差）
        numeric_cols = ['开盘价', '最高价', '最低价', '收盘价', '成交量']
        for col in numeric_cols:
            df = df[(np.abs(stats.zscore(df[col])) < 3)]
        
        # 3. 去除停牌数据（成交量为0）
        df = df[df['成交量'] > 0]
        
        # 4. 重置索引
        self.clean_data = df.reset_index(drop=True)
        return self.clean_data

# ===================== 2. 因子构建 =====================
class FactorConstructor:
    def __init__(self, clean_data):
        self.data = clean_data
        self.factors = None

    # 构建基础因子
    def build_factors(self):
        df = self.data.copy()
        
        # 按股票分组计算因子
        factor_df = df.groupby('code').apply(self._calculate_single_stock_factors).reset_index(drop=True)
        
        # 因子去极值（MAD法）
        factor_cols = ['return_5d', 'return_20d', 'volatility_20d', 'ma5_ma20', 'turnover_20d']
        for col in factor_cols:
            factor_df[col] = self._winsorize(factor_df[col])
        
        # 因子标准化（Z-score）
        scaler = StandardScaler()
        factor_df[factor_cols] = scaler.fit_transform(factor_df[factor_cols])
        
        # 行业中性化（简化版：按市值分组）
        # 实际应用中应使用申万/中信行业分类
        factor_df['size_group'] = pd.qcut(factor_df['收盘价'], 5, labels=False)
        for col in factor_cols:
            factor_df[col] = factor_df.groupby(['日期', 'size_group'])[col].transform(
                lambda x: x - x.mean()
            )
        
        self.factors = factor_df
        return self.factors

    # 单只股票因子计算
    def _calculate_single_stock_factors(self, df):
        df = df.sort_values('日期')
        
        # 1. 收益率因子
        df['return_5d'] = df['收盘价'].pct_change(5)
        df['return_20d'] = df['收盘价'].pct_change(20)
        
        # 2. 波动率因子
        df['volatility_20d'] = df['收盘价'].pct_change().rolling(20).std()
        
        # 3. 均线因子
        df['ma5'] = df['收盘价'].rolling(5).mean()
        df['ma20'] = df['收盘价'].rolling(20).mean()
        df['ma5_ma20'] = (df['ma5'] - df['ma20']) / df['ma20']
        
        # 4. 换手率因子
        df['turnover_20d'] = df['成交量'].rolling(20).mean() / df['成交量'].rolling(60).mean()
        
        return df

    # 因子去极值
    def _winsorize(self, series, n=3):
        median = series.median()
        mad = np.median(np.abs(series - median))
        upper = median + n * mad
        lower = median - n * mad
        return series.clip(lower, upper)

# ===================== 3. 机器学习预测 =====================
class MLForecaster:
    def __init__(self, factor_data):
        self.data = factor_data
        self.model = None
        self.predictions = None

    # 准备预测数据
    def prepare_prediction_data(self):
        df = self.data.copy()
        
        # 预测目标：未来5日收益率
        df['target'] = df.groupby('code')['收盘价'].pct_change(5).shift(-5)
        
        # 去除缺失值
        self.pred_data = df.dropna(subset=['return_5d', 'return_20d', 'volatility_20d', 
                                          'ma5_ma20', 'turnover_20d', 'target'])
        return self.pred_data

    # 训练模型并预测
    def train_and_predict(self):
        # 特征和目标变量
        X = self.pred_data[['return_5d', 'return_20d', 'volatility_20d', 
                           'ma5_ma20', 'turnover_20d']]
        y = self.pred_data['target']
        
        # 划分训练集和测试集
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )
        
        # 训练随机森林模型
        self.model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
        self.model.fit(X_train, y_train)
        
        # 预测
        self.pred_data['pred_return'] = self.model.predict(X)
        
        # 按预测收益率排序
        self.pred_data['rank'] = self.pred_data.groupby('日期')['pred_return'].rank(ascending=False)
        self.predictions = self.pred_data
        return self.predictions

# ===================== 4. 组合优化 =====================
class PortfolioOptimizer:
    def __init__(self, prediction_data):
        self.data = prediction_data
        self.weights = None

    # 均值-方差优化
    def optimize_portfolio(self):
        # 按日期分组优化
        self.weights = []
        
        for date, group in self.data.groupby('日期'):
            # 只选预测收益率前20%的股票
            top_stocks = group[group['rank'] <= len(group) * 0.2]
            
            if len(top_stocks) < 5:
                continue
                
            # 预期收益和协方差矩阵
            expected_returns = top_stocks['pred_return'].values
            returns_matrix = top_stocks[['return_5d', 'return_20d']].values
            cov_matrix = np.cov(returns_matrix.T)
            
            # 优化目标：最大化夏普比率
            n = len(expected_returns)
            w = cp.Variable(n)
            
            # 约束条件
            constraints = [
                cp.sum(w) == 1,          # 权重和为1
                w >= 0,                  # 不做空
                w <= 0.1                 # 单票权重不超过10%
            ]
            
            # 目标函数
            risk = cp.quad_form(w, cov_matrix)
            return_obj = w @ expected_returns
            objective = cp.Maximize(return_obj / cp.sqrt(risk + 1e-6))  # 加小值避免除零
            
            # 求解
            problem = cp.Problem(objective, constraints)
            try:
                problem.solve()
                if w.value is not None:
                    weights_df = pd.DataFrame({
                        '日期': date,
                        'code': top_stocks['code'].values,
                        'weight': w.value
                    })
                    self.weights.append(weights_df)
            except:
                # 优化失败则等权
                equal_weight = 1 / len(top_stocks)
                weights_df = pd.DataFrame({
                    '日期': date,
                    'code': top_stocks['code'].values,
                    'weight': equal_weight
                })
                self.weights.append(weights_df)
        
        self.weights = pd.concat(self.weights, ignore_index=True)
        return self.weights

# ===================== 5. 回测框架 =====================
class MultiFactorStrategy(bt.Strategy):
    params = (
        ('rebalance_days', 5),  # 5个交易日调仓一次
    )

    def __init__(self, weights_data):
        self.weights = weights_data
        self.rebalance_day = 0
        self.current_weights = {}

    def next(self):
        # 定期调仓
        if self.rebalance_day % self.params.rebalance_days == 0:
            self.rebalance_portfolio()
        self.rebalance_day += 1

    def rebalance_portfolio(self):
        # 获取当前日期的权重
        current_date = self.datas[0].datetime.date(0)
        date_str = current_date.strftime('%Y-%m-%d')
        
        try:
            day_weights = self.weights[self.weights['日期'] == date_str]
        except:
            return
        
        # 平仓不在目标组合中的股票
        for data in self.datas:
            if data._name not in day_weights['code'].values and self.getposition(data).size > 0:
                self.close(data)
        
        # 开仓目标股票
        total_value = self.broker.getvalue()
        for _, row in day_weights.iterrows():
            code = row['code']
            weight = row['weight']
            target_value = total_value * weight
            
            for data in self.datas:
                if data._name == code:
                    current_pos = self.getposition(data).size
                    target_pos = int(target_value / data.close[0])
                    
                    if target_pos > current_pos:
                        self.buy(data, size=target_pos - current_pos)
                    elif target_pos < current_pos:
                        self.sell(data, size=current_pos - target_pos)
                    break

# ===================== 6. 主执行流程 =====================
def main():
    # 1. 数据处理
    print("=== 数据获取与清洗 ===")
    dp = DataProcessor()
    dp.get_stock_list()
    dp.get_stock_data(start_date="2020-01-01", end_date="2024-01-01")
    clean_data = dp.clean_data()
    print(f"清洗后数据量：{len(clean_data)}")

    # 2. 因子构建
    print("\n=== 因子构建 ===")
    fc = FactorConstructor(clean_data)
    factor_data = fc.build_factors()
    print(f"因子数据量：{len(factor_data)}")
    print("因子列表：", [col for col in factor_data.columns if col not in clean_data.columns])

    # 3. 机器学习预测
    print("\n=== 机器学习预测 ===")
    ml = MLForecaster(factor_data)
    pred_data = ml.prepare_prediction_data()
    pred_results = ml.train_and_predict()
    print(f"预测数据量：{len(pred_results)}")
    print(f"模型特征重要性：{ml.model.feature_importances_}")

    # 4. 组合优化
    print("\n=== 组合优化 ===")
    po = PortfolioOptimizer(pred_results)
    weights = po.optimize_portfolio()
    print(f"优化后权重数据量：{len(weights)}")

    # 5. 回测准备
    print("\n=== 策略回测 ===")
    cerebro = bt.Cerebro()
    
    # 添加数据
    for code in dp.stock_codes[:10]:
        stock_data = clean_data[clean_data['code'] == code].copy()
        stock_data = stock_data.rename(columns={
            '日期': 'datetime', '开盘价': 'open', '最高价': 'high',
            '最低价': 'low', '收盘价': 'close', '成交量': 'volume'
        })
        stock_data = stock_data.set_index('datetime')
        
        data_feed = bt.feeds.PandasData(
            dataname=stock_data,
            name=code,
            datetime=None,
            open='open',
            high='high',
            low='low',
            close='close',
            volume='volume',
            openinterest=None
        )
        cerebro.adddata(data_feed)

    # 添加策略
    cerebro.addstrategy(MultiFactorStrategy, weights_data=weights)
    
    # 设置初始资金
    cerebro.broker.setcash(1000000.0)
    # 设置手续费
    cerebro.broker.setcommission(commission=0.0005)
    
    # 添加分析器
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    # 运行回测
    print(f"初始资金：{cerebro.broker.getvalue():.2f}")
    results = cerebro.run()
    print(f"最终资金：{cerebro.broker.getvalue():.2f}")
    
    # 打印绩效指标
    strat = results[0]
    print(f"总收益率：{strat.analyzers.returns.get_analysis()['rtot'] * 100:.2f}%")
    print(f"夏普比率：{strat.analyzers.sharpe.get_analysis()['sharperatio']:.2f}")
    print(f"最大回撤：{strat.analyzers.drawdown.get_analysis()['max']['drawdown']:.2f}%")
    
    # 绘制回测结果
    cerebro.plot(style='candlestick')

if __name__ == "__main__":
    main()